# 03 — Two-Stage Default · CLF · xgb

Stage 1 분류 (binary, `y_unit > 0`) → die-level prob csv → combine 단계에서 reg와 곱.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/clf/xgb/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` (strategy_common §1)
- **HPO**: N_TRIALS=1, **Wide search** (1차는 1 trial만이라 anchor 신뢰도 낮음 — strategy.md §3.2)
- **Sampler**: TPE seed=None multivariate group (§4)
- **Pruner**: MedianPruner (§4)
- **Timeout**: TIMEOUT_SEC (§25)


## 1. 환경 설정 + import

In [1]:
import os, sys

GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'  # ★ Colab 사용 시 신규 modeling.zip ID 입력

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo, models
from meta_features import add_meta_features   # 2_preprocessing/meta_features.py — 2026-05-09 결정 반영

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
optuna v4.7.0


## 2. 실험 설정

In [2]:
# ── 모델 ──
CLF_MODEL_NAME = 'xgb'
assert CLF_MODEL_NAME in models.CLF_AVAILABLE_MODELS

# ── 실험 식별 ──
EXP_ID   = f'ts-clf-{CLF_MODEL_NAME}-002'
EXP_MEMO = f'Two-Stage default · CLF · {CLF_MODEL_NAME} · wide search'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS         = 3000
N_FOLDS          = 5
N_STARTUP_TRIALS = 50
N_JOBS           = 7   # ★ strategy_common §8 (병렬 2 노트북). 단독 실행 시 14
TIMEOUT_SEC      = 90 * 60 * 60  # ★ Colab 타임아웃 대비, 초 단위 (None=무제한)

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'clf', CLF_MODEL_NAME, EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 트리 PP_FIXED (strategy_common §1) ──
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# ── 손실 ──
CLIP_Y_EXTREME = True

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'CLF_MODEL_NAME: {CLF_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'OUT_DIR={OUT_DIR}')


EXP: ts-clf-xgb-002 | USER: jh
CLF_MODEL_NAME: xgb
N_TRIALS=1 | N_FOLDS=5 | N_JOBS=7 | TIMEOUT_SEC=None
OUT_DIR=C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\clf\xgb


## 3. 데이터 로드 + Y clip + 전처리

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# ── 메타피처 추가 (2026-05-09 결정: 트리 분류=position raw + die_xy continuous) ──
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

y_train_unit = ys_input['train']
n_pos = (y_train_unit[TARGET_COL] > 0).sum()
n_neg = (y_train_unit[TARGET_COL] == 0).sum()
print(f'Unit binary 분포: pos={n_pos:,} ({n_pos/(n_pos+n_neg):.1%}), neg={n_neg:,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행


[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572


[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729


xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031


[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개


...


  [요약] 343,494 → 공간(161,870) → lot(100,428) → 전체(81,196) → 잔여(0)


[고상관 제거] threshold=0.96, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.96
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 633)

클리닝 완료: 1031 → 564 features (467개 제거)
  + indicator 컬럼: 9개 → 총 573개
  train: (104748, 633)
  val:   (34908, 633)
  test:  (34916, 633)
이상치 처리 파이프라인 시작 (method=winsorize)


[이상치 탐지] IQR × 1.5
  이상치 > 5%: 112개
  이상치 > 10%: 64개


[Winsorization] lower=0%, upper=99%
  적용 feature: 573개

이상치 처리 완료 (method=winsorize)
  train: (104748, 633)


[add_meta_features] position_mode='raw', use_die_xy=True, use_loc_x_ohe=False → position=['position'], die_xy=['die_x', 'die_y'] (feat_cols: 576)

[전처리 완료] feat_cols: 576
Unit binary 분포: pos=7,646 (29.2%), neg=18,541


## 4. Optuna HPO

- **Sampler**: `TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)`
- **Pruner**: `MedianPruner(n_warmup_steps=10)`
- **Timeout**: `TIMEOUT_SEC` (None=무제한)
- anchor enqueue **없음** (1차 1 trial 신뢰도 낮음)

In [4]:
study_meta = {
    'exp_id':           EXP_ID,
    'exp_memo':         EXP_MEMO,
    'user':             USER,
    'clf_model_name':   CLF_MODEL_NAME,
    'clip_y_extreme':   CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':         N_TRIALS,
    'n_folds':          N_FOLDS,
    'n_jobs':           N_JOBS,
    'timeout_sec':      TIMEOUT_SEC,
    'seed':             SEED,
    'sampler':          'TPE seed=None multivariate group',
    'pruner':           f'MedianPruner n_warmup=10',
}

sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
pruner  = MedianPruner(n_warmup_steps=10)

res = hpo.run_clf_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=CLF_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    user_attrs=study_meta,
    sampler=sampler,
    pruner=pruner,
    enqueue_trials=None,         # ★ anchor enqueue 없음 (1차 1 trial 신뢰도 낮음, strategy.md §3.2)
    timeout=TIMEOUT_SEC,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']
y_pos_const = res['y_pos_const']

print(f'\n[HPO 완료] best train RMSE = {res["best_value"]:.6f}')
print(f'y_pos_const (E[Y|Y>0]) = {y_pos_const:.6f}')
print(f'best_params = {best_params}')


  0%|          | 0/1 [00:00<?, ?it/s]


[HPO 완료] best train RMSE = 0.005717
y_pos_const (E[Y|Y>0]) = 0.008496
best_params = {'n_estimators': 1362, 'learning_rate': 0.01692341182512994, 'max_depth': 9, 'min_child_weight': 17.41066902828611, 'subsample': 0.5546714783138929, 'colsample_bytree': 0.6339022027675953, 'reg_alpha': 0.00016207834494842664, 'reg_lambda': 0.023539737828662113, 'gamma': 0.00014175461145660208, 'scale_pos_weight': 3.5}


## 5. Best trial 재학습 (K-fold OOF) + die-level prob 캐쳐

In [5]:
final = hpo.refit_clf_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=CLF_MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
)

def _unit_mean_proba(xs_split, die_proba):
    df = pd.DataFrame({KEY_COL: xs_split[KEY_COL].values, 'p': die_proba})
    return df.groupby(KEY_COL, sort=False)['p'].mean()

def _rmse(pred_unit, y_unit_df):
    aligned = pred_unit.loc[y_unit_df.set_index(KEY_COL).index]
    return float(np.sqrt(np.mean((aligned.values - y_unit_df[TARGET_COL].values) ** 2)))

oof_unit_proba  = _unit_mean_proba(xs_train, final['oof_proba_die'])
val_unit_proba  = _unit_mean_proba(xs_val,   final['val_proba_die'])
test_unit_proba = _unit_mean_proba(xs_test,  final['test_proba_die'])

oof_rmse  = _rmse(oof_unit_proba * y_pos_const, ys_input['train'])
val_rmse  = _rmse(val_unit_proba * y_pos_const, ys_input['validation'])
test_rmse = _rmse(test_unit_proba * y_pos_const, ys_input['test'])

print(f'\n[Refit 완료] (clf 단독, prob × y_pos_const)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')


[clf refit fold 1/5] tr_units=20949, vl_units=5238, pos_ratio=0.290


[clf refit fold 2/5] tr_units=20949, vl_units=5238, pos_ratio=0.294


[clf refit fold 3/5] tr_units=20950, vl_units=5237, pos_ratio=0.293


[clf refit fold 4/5] tr_units=20950, vl_units=5237, pos_ratio=0.292


[clf refit fold 5/5] tr_units=20950, vl_units=5237, pos_ratio=0.291

[Refit 완료] (clf 단독, prob × y_pos_const)
  OOF  unit RMSE = 0.005717
  val  unit RMSE = 0.005904
  test unit RMSE = 0.008519


## 6. 산출물 저장

In [6]:
hpo.save_clf_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    y_pos_const=y_pos_const,
    study_meta=study_meta,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

# Colab → 로컬 zip 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'clf_{CLF_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass


[save_clf_artifacts] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\clf\xgb 저장 완료 (fold_models.pkl + best_params.json + 6 CSV)
  best_params.json                       9.8 KB
  fold_models.pkl                   57,659.0 KB
  oof_die.csv                        5,340.3 KB
  oof_unit.csv                       1,453.3 KB
  optuna_jh_ts-clf-xgb-002.db          112.0 KB
  test_die.csv                       1,779.5 KB
  test_unit.csv                        484.3 KB
  val_die.csv                        1,779.6 KB
  val_unit.csv                         484.2 KB
